<a href="https://colab.research.google.com/github/Delusional-bishop/Fortification-App/blob/main/Fortification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ✅ 1. Install Ultralytics (YOLOv5/YOLOv8)
!pip install -q ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.7 MB/s eta 0:00:00


In [ ]:
! gdown --id 1vKx6nFaS7srv2Ui74NBW4dM9qtuT_Mr1

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1vKx6nFaS7srv2Ui74NBW4dM9qtuT_Mr1
From (redirected): https://drive.google.com/uc?id=1vKx6nFaS7srv2Ui74NBW4dM9qtuT_Mr1&confirm=t&uuid=f72db4c1-0a73-414b-8cc2-9a62b21686fa
To: /content/dataaa.zip
100% 73.0M/73.0M [00:01<00:00, 44.1MB/s]


In [ ]:
# ✅ 2. Unzip Your Dataset
!unzip -q /content/dataaa.zip -d /content/custom_data
#

In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import yaml


In [ ]:
from pathlib import Path
import random
import os
import shutil

# === CONFIGURATION ===
data_path = "/content/custom_data"  # Root folder containing "images" and "labels"
train_percent = 0.9  # Fraction of data to use for training

# === VALIDATION ===
if not os.path.isdir(data_path):
    raise ValueError("Error: Directory specified in data_path does not exist.")

if train_percent < 0.01 or train_percent > 0.99:
    raise ValueError("Error: train_percent must be between 0.01 and 0.99.")

# === PATHS ===
input_image_path = os.path.join(data_path, 'images')
input_label_path = os.path.join(data_path, 'labels')

output_base = '/content/data'
train_img_path = os.path.join(output_base, 'train/images')
train_txt_path = os.path.join(output_base, 'train/labels')
val_img_path = os.path.join(output_base, 'validation/images')
val_txt_path = os.path.join(output_base, 'validation/labels')

# === CREATE OUTPUT FOLDERS ===
for dir_path in [train_img_path, train_txt_path, val_img_path, val_txt_path]:
    os.makedirs(dir_path, exist_ok=True)

# === GET IMAGE FILES ===
img_file_list = list(Path(input_image_path).rglob('*'))
print(f"Total image files found: {len(img_file_list)}")

# === SPLIT DATA ===
random.shuffle(img_file_list)
train_num = int(len(img_file_list) * train_percent)
train_files = img_file_list[:train_num]
val_files = img_file_list[train_num:]

print(f"Images to train: {len(train_files)}")
print(f"Images to validation: {len(val_files)}")

# === COPY IMAGES AND LABELS ===
def copy_files(file_list, dest_img_path, dest_txt_path):
    for img_path in file_list:
        img_fn = img_path.name
        base_fn = img_path.stem
        txt_path = os.path.join(input_label_path, base_fn + '.txt')

        # Copy image
        shutil.copy(img_path, os.path.join(dest_img_path, img_fn))

        # Copy label if it exists
        if os.path.exists(txt_path):
            shutil.copy(txt_path, os.path.join(dest_txt_path, base_fn + '.txt'))

copy_files(train_files, train_img_path, train_txt_path)
copy_files(val_files, val_img_path, val_txt_path)

print("✅ Dataset split complete.")


Total image files found: 400
Images to train: 360
Images to validation: 40
✅ Dataset split complete.


In [ ]:
# ✅ 5. Generate data.yaml File
def create_data_yaml(classes_txt_path, output_yaml_path, data_root='/content/data'):
    if not os.path.exists(classes_txt_path):
        raise FileNotFoundError(f"{classes_txt_path} not found!")

    with open(classes_txt_path) as f:
        classes = [line.strip() for line in f if line.strip()]

    data_yaml = {
        'path': data_root,
        'train': 'train/images',
        'val': 'validation/images',
        'nc': len(classes),
        'names': classes
    }

    with open(output_yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, sort_keys=False)

    print(f"✅ Created config at {output_yaml_path}")

# Create the YAML file
create_data_yaml(
    classes_txt_path="/content/custom_data/classes.txt",
    output_yaml_path="/content/data.yaml"
)


✅ Created config at /content/data.yaml


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # Use yolov5s.pt only if you're using YOLOv5 repo directly
model.train(
    data="/content/data.yaml",
    epochs=150,
    imgsz=640,
    batch=32,
    name="rice_yolov8n",
    cache=True
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 49.7M/49.7M [00:00<00:00, 127MB/s]


Ultralytics 8.3.162 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=rice_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=Tr

100%|██████████| 755k/755k [00:00<00:00, 22.9MB/s]

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     13/150      12.5G      1.402      0.612     0.9403        224        640: 100%|██████████| 12/12 [00:11<00:00,  1.00it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  2.67it/s]

                   all         40        381      0.726      0.628      0.632      0.319



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     14/150      12.6G      1.453     0.6125      0.962        528        640:   8%|▊         | 1/12 [00:01<00:11,  1.02s/it]

In [ ]:
# a


In [ ]:
!yolo detect predict model=runs/detect/rice_yolov8n7/weights/best.pt source=data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:20]:
  display(Image(filename=image_path, height=400))
  print('\n')


In [ ]:
# Create "my_model" folder to store model weights and train results
!mkdir /content/my_model
!cp /content/runs/detect/rice_yolov8n7/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/rice_yolov8n7 /content/my_model

# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip rice_yolov8n7
%cd /content

mkdir: cannot create directory ‘/content/my_model’: File exists
/content/my_model
updating: my_model.pt (deflated 8%)
  adding: rice_yolov8n7/ (stored 0%)
  adding: rice_yolov8n7/weights/ (stored 0%)
  adding: rice_yolov8n7/weights/last.pt (deflated 8%)
  adding: rice_yolov8n7/weights/best.pt (deflated 8%)
  adding: rice_yolov8n7/val_batch0_labels.jpg (deflated 7%)
  adding: rice_yolov8n7/train_batch0.jpg (deflated 6%)
  adding: rice_yolov8n7/BoxF1_curve.png (deflated 17%)
  adding: rice_yolov8n7/results.csv (deflated 64%)
  adding: rice_yolov8n7/args.yaml (deflated 53%)
  adding: rice_yolov8n7/labels_correlogram.jpg (deflated 37%)
  adding: rice_yolov8n7/confusion_matrix_normalized.png (deflated 31%)
  adding: rice_yolov8n7/train_batch1820.jpg (deflated 20%)
  adding: rice_yolov8n7/train_batch2.jpg (deflated 6%)
  adding: rice_yolov8n7/val_batch1_pred.jpg (deflated 6%)
  adding: rice_yolov8n7/train_batch1.jpg (deflated 8%)
  adding: rice_yolov8n7/train_batch1822.jpg (deflated 20%)
  a